# Spatial CNN feature-reconstruction anomaly detector

## PerPixelReconstructor vs SpatialReconstructor

### What changed and what motivated it
**Spatial reconstruction:** the 3x3 layers use neighbouring features, reducing false positives caused by normal local variation in parts of cables, capsules, transistor,etc


**Balanced feature levels:** L2-normalizing layer 2 and layer 3 features prevents one level's activation scale from dominating the loss and forces autoencoder to learn reconstructing features of both layers and not just the one with higher scale

**Random Affine Transform to training dataset:** This reduces sensitivity of model towards slight variations in absolute pixel positions of object 

**Robust scoring:** Smoothing plus top-1% image scoring to improve image AUROC where earlier a max score was triggered by a single mismatch.


**Correct masks:** nearest-neighbour resizing prevents evaluation labels from becoming fractional at boundaries.

In general the performance is expected to be better compared to PerPixelReconstructor

The detector is still trained separately per category and only on train/good, which prevents leakage from defective test images.

In [1]:
import os
import glob
import random
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import ResNet50_Weights

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## Data Loading and Processing

In [2]:
class MVTecDataset(Dataset):
    def __init__(self,root_dir,category,is_train=True,transform=None,mask_transform=None):
        self.transform = transform
        self.mask_transform = mask_transform
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        root_dir = Path(root_dir)
        
        if is_train:
            img_dir = root_dir/category/"train"/"good"
            paths = sorted(img_dir.glob("*.png"))
            self.image_paths.extend(paths)
            self.labels.extend([0]*len(paths)) # 0 for no defect
            self.mask_paths.extend([None]*len(paths)) # No mask for non defective
        
        else:
            test_dir = root_dir/category/"test"
            for defect in sorted(os.listdir(test_dir)):
                if defect.startswith('.'):
                    continue
                paths = sorted((test_dir/defect).glob("*.png"))
                self.image_paths.extend(paths)
                
                if defect == "good":
                    self.labels.extend([0]*len(paths))
                    self.mask_paths.extend([None]*len(paths))
                else:
                    self.labels.extend([1]*len(paths))
                    for p in paths:
                        mask_path = root_dir/category/"ground_truth"/defect/f"{p.stem}_mask.png"
                        self.mask_paths.append(mask_path)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self,idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        mask_path = self.mask_paths[idx]
        
        if mask_path and mask_path.exists():
            mask = Image.open(mask_path).convert("L")
            if self.mask_transform:
                mask = self.mask_transform(mask)
        else:
            mask = torch.zeros((1,image.shape[1],image.shape[2]),dtype=torch.float32)
        return image,label,mask

## Feature Extraction from ResNet Layers

Features from ResNet layers 2 and 3 are combined: layer 2 preserves relatively fine defect detail while layer 3 supplies more semantic context. Each layer is L2-normalized before concatenation so the deeper layer cannot dominate the reconstruction loss merely because of its scale.


Layer 2 : 512 Channels

Layer 3 : 1024 Channels

Layer 3 has lower spatial resolution, so it is bilinearly resized to layer 2 before concatenation.

In [3]:
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        
        # 512 Channels
        self.encoder_l2 = nn.Sequential(
            resnet.conv1,resnet.bn1,resnet.relu,resnet.maxpool,resnet.layer1,resnet.layer2
        )
        
        # 1024 Channels
        self.encoder_l3 = resnet.layer3
        
        for param in self.parameters():
            param.requires_grad = False

    def forward(self,x):
        with torch.no_grad():
            feat2 = self.encoder_l2(x) # Larger Image
            feat3 = self.encoder_l3(feat2) # Smaller Image : Needs to be resized before concat
            
            feat3_up = F.interpolate(feat3,size=feat2.shape[2:],mode="bilinear",align_corners=False)
            
            # L2 regularise both features to prevent differences in impact solely due to scale of layer activations
            feat2 = F.normalize(feat2,p=2,dim=1)
            feat3_up = F.normalize(feat3_up,p=2,dim=1)
            return torch.cat([feat2,feat3_up],dim=1)

## Spatial convolutional autoencoder

The baseline used only 1x1 convolutions, so every position was reconstructed independently. This encoder includes 3x3 convolutions, letting it compare a feature with its local neighbourhood : important for contours, small parts and repeating textures. Group normalization avoids the batch size sensitivity of batch normalization.

A 128-channel bottleneck limits capacity while the spatial layers model normal local structure.

Trained only on train/good so that model learns to encode and decode perfect textures

An anomalous local pattern should have a larger reconstruction error than normal training features.

The resulting error map is lightly smoothed and image-level scoring uses the mean of the highest-scoring 1% of locations instead of a single-pixel maximum.

In [4]:
class ConvAutoEncoder(nn.Module):
    def __init__(self,in_channels=1536):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,out_channels=512,kernel_size=1,bias=False),
            nn.GroupNorm(num_groups=32,num_channels=512),
            nn.GELU(),
            nn.Conv2d(in_channels=512,out_channels=128,kernel_size=3,padding=1,bias=False),
            nn.GroupNorm(num_groups=16,num_channels=128),
            nn.GELU()
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels=128,out_channels=512,kernel_size=3,padding=1,bias=False),
            nn.GroupNorm(num_groups=32,num_channels=512),
            nn.GELU(),
            nn.Conv2d(in_channels=512,out_channels=in_channels,kernel_size=1)
        )
    
    def forward(self,x):
        return self.decoder(self.encoder(x))

In [5]:
def train_evaluate(category="bottle",root_dir="./datasets/MVTecAD",epochs=40,batch_size=32,topk_fraction=0.01):
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    
    img_size = 256
    transform = transforms.Compose([
        transforms.Resize((img_size,img_size),antialias=True),
        transforms.RandomAffine(degrees=2, translate=(0.03, 0.03)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
    ])
    mask_transform = transforms.Compose([
        # Masks are categorical labelsv : nearest-neighbour avoids creating soft defect borders and instead creates strict 0/1 boundaries for defects
        transforms.Resize((img_size,img_size),interpolation=InterpolationMode.NEAREST),
        transforms.ToTensor()
    ])
    
    train_dataset = MVTecDataset(root_dir,category,is_train=True,transform=transform)
    test_dataset = MVTecDataset(root_dir,category,is_train=False,transform=transform,mask_transform=mask_transform)
    
    train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    test_loader = DataLoader(test_dataset,batch_size=1,shuffle=False)
    
    extractor = ResNetFeatureExtractor().to(device).eval()
    autoencoder = ConvAutoEncoder().to(device)
    optimizer = torch.optim.AdamW(autoencoder.parameters(),lr=3e-4,weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)
    
    print(f"Autoencoder training for {category.capitalize()}")
    autoencoder.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for images,labels,masks in train_loader:
            images = images.to(device)
            features = extractor(images)
            
            optimizer.zero_grad()
            reconstr = autoencoder(features)
            loss = F.mse_loss(reconstr,features)
            loss.backward()
            optimizer.step()
            
            epoch_loss+=loss.item()
            
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} | MSE: {epoch_loss/len(train_loader):.5f}")
    
    print(f"Evaluating {category.capitalize()}")
    autoencoder.eval()
    
    image_scores,image_labels = [],[]
    # A top-k mean is much less vulnerable to a single noisy feature location than max() (Similar to KNN vs NN)
    
    pixel_scores,pixel_labels = [],[]
    # pixel_score is pixel by pixel squared difference flatten
    # pixel_label is pixel by pixel mask image flattened (0 for good part, 1 for defective portion)

    with torch.no_grad():
        for images,labels,masks in test_loader:
            images = images.to(device)
            features = extractor(images)
            reconstr = autoencoder(features)
            
            mse_map = torch.mean((features-reconstr)**2,dim=1,keepdim=True)
            mse_map = F.avg_pool2d(mse_map,kernel_size=3,stride=1,padding=1)
            
            # Ground_Truth is 256x256 but mse_map is only 32x32
            anomaly_map = F.interpolate(mse_map,size=(img_size,img_size),mode="bilinear",align_corners=False)
            anomaly_map = anomaly_map.cpu().squeeze().numpy()
            mask = masks.squeeze().numpy()
            
            topk = max(1,int(anomaly_map.size*topk_fraction))
            image_scores.append(np.partition(anomaly_map.ravel(),-topk)[-topk:].mean()) # Mean of top k values across all pixels
            image_labels.append(labels.item()) 
            
            pixel_scores.extend(anomaly_map.flatten())
            pixel_labels.extend((mask.flatten()>0).astype(int))
            
    img_auroc = roc_auc_score(image_labels,image_scores)
    pxl_auroc = roc_auc_score(pixel_labels,pixel_scores)
    
    results = {"image_auroc": img_auroc, "pixel_auroc": pxl_auroc}
    print(f"[{category.upper()}] Image AUROC: {img_auroc:.4f} | Pixel AUROC: {pxl_auroc:.4f}")
    return results

**Interpretation of Evaluation**

mse_map is the mean squared reconstruction error across feature channels. A 3x3 average pool suppresses isolated feature noise before the map is resized to the input resolution.

Ground truth is 256x256 while mse_map is only 32x32, so we interpolate back to 256x256 bilinearly (anomaly_map)

image_score is the mean of the largest 1% of values in the anomaly map. This retains sensitivity to compact defects while being substantially less susceptible to a single false-positive location than np.max().
image_label is the 0/1 label for good/defective object

pixel_score is the flattened anomaly map. Ground-truth masks are resized with nearest-neighbour interpolation, preserving binary defect boundaries.

pixel_label is the the flatten mask image (after scaling pixels to 0-1) : So defective portion's pixels are 1 and normal pixels are 0

## Performance

In [6]:
if __name__ == "__main__":
    target_categories = ["bottle","cable","capsule","carpet","hazelnut","transistor","wood"]

    for cat in target_categories:
        print("\n" + "-"*50)
        print(f"Starting Pipeline for : {cat.capitalize()}")
        print("-"*50)
        
        train_evaluate(category=cat,root_dir="./datasets/MVTecAD",epochs=40)
        print()


--------------------------------------------------
Starting Pipeline for : Bottle
--------------------------------------------------
Autoencoder training for Bottle
Epoch 1/40 | MSE: 0.05876
Epoch 2/40 | MSE: 0.00669
Epoch 3/40 | MSE: 0.00416
Epoch 4/40 | MSE: 0.00224
Epoch 5/40 | MSE: 0.00163
Epoch 6/40 | MSE: 0.00131
Epoch 7/40 | MSE: 0.00112
Epoch 8/40 | MSE: 0.00102
Epoch 9/40 | MSE: 0.00096
Epoch 10/40 | MSE: 0.00091
Epoch 11/40 | MSE: 0.00089
Epoch 12/40 | MSE: 0.00086
Epoch 13/40 | MSE: 0.00085
Epoch 14/40 | MSE: 0.00083
Epoch 15/40 | MSE: 0.00082
Epoch 16/40 | MSE: 0.00081
Epoch 17/40 | MSE: 0.00080
Epoch 18/40 | MSE: 0.00079
Epoch 19/40 | MSE: 0.00078
Epoch 20/40 | MSE: 0.00078
Epoch 21/40 | MSE: 0.00077
Epoch 22/40 | MSE: 0.00077
Epoch 23/40 | MSE: 0.00076
Epoch 24/40 | MSE: 0.00076
Epoch 25/40 | MSE: 0.00076
Epoch 26/40 | MSE: 0.00075
Epoch 27/40 | MSE: 0.00075
Epoch 28/40 | MSE: 0.00075
Epoch 29/40 | MSE: 0.00075
Epoch 30/40 | MSE: 0.00075
Epoch 31/40 | MSE: 0.00075
Epoch 